# Vector Database Landscape and Trade-offs

Chroma is the right place to **learn** vector storage — but production teams pick among embedded libraries, SQL extensions, self-hosted servers, and managed clouds. The best choice depends on **scale**, **ops capacity**, **filtering**, **hybrid search**, **cost**, and what you already run (especially PostgreSQL).

This notebook maps the vector database landscape, compares major systems with trade-off tables, provides a structured **decision framework**, explains **hybrid search** and reciprocal rank fusion, shows a **portable VectorStore** interface, and walks through migration paths from prototype to production.

Prerequisites: **01–07** in this section. No multi-vendor setup required — evaluate vendors using **09. Evaluating Retrieval Quality** on your data.

In [ ]:
from dataclasses import dataclass
from typing import Any, Protocol

---

## 1. The Vector Storage Spectrum

```
  Simple / local                              Enterprise / managed
  ─────────────────────────────────────────────────────────────►
  NumPy    FAISS    Chroma    LanceDB    pgvector    Qdrant    Pinecone    Milvus
```

| Category | Examples | You manage | Typical scale |
|----------|----------|------------|---------------|
| **In-process** | NumPy, FAISS | Memory, files | Research, < 1M |
| **Embedded DB** | Chroma, LanceDB | Disk path, backups | Dev → mid-size |
| **SQL extension** | pgvector, SQLite-vec | Your Postgres | App data + vectors |
| **Vector server** | Qdrant, Weaviate, Milvus | Cluster, upgrades | Large self-hosted |
| **Managed SaaS** | Pinecone, Weaviate Cloud | Billing, API keys | Scale without ops |

**Insight:** All implement the same idea — store vectors, run ANN search, attach metadata — but differ in **ops**, **filter expressiveness**, and **hybrid features**.

---

## 2. System-by-System Overview

### 2.1 Chroma (This Curriculum)

| Strength | Limitation |
|----------|------------|
| Fast local dev, simple Python API | Not a full enterprise story alone |
| Persistent client, metadata filters | Very large scale may need migration |
| LangChain integration | Fewer managed SLAs than cloud vendors |

**Best for:** Learning, prototypes, small/medium RAG apps, single-node deployments.

### 2.2 FAISS (Facebook AI Similarity Search)

Library, not a database — indexes in memory or on disk; **you** build metadata and filtering in application code.

| Strength | Limitation |
|----------|------------|
| Extremely fast ANN research baseline | No CRUD API, no server |
| GPU indexes available | Manual metadata/filter layer |

**Best for:** Offline benchmarks, custom pipelines, ML research.

### 2.3 pgvector (PostgreSQL Extension)

Store vectors in a `vector` column; query with SQL + distance operators.

| Strength | Limitation |
|----------|------------|
| One database for app + vectors + joins | ANN tuning less turnkey than dedicated DBs |
| Transactions, backups, familiar ops | Very high QPS may need dedicated store |

**Best for:** Teams already on Postgres; moderate vector scale; strong consistency needs.

### 2.4 Pinecone (Managed)

Fully managed vector index — no cluster to operate.

| Strength | Limitation |
|----------|------------|
| Low ops, scales with usage | Cost at high volume; vendor lock-in |
| Namespaces, metadata filters | Data leaves your VPC (unless enterprise) |

**Best for:** Teams without vector ops expertise needing reliable scale.

### 2.5 Weaviate

Open-source vector DB with optional modules (vectorization, hybrid BM25).

| Strength | Limitation |
|----------|------------|
| Hybrid search built-in | More complex deployment self-hosted |
| GraphQL API, modules | Learning curve vs Chroma |

**Best for:** Hybrid keyword+vector search, modular ingestion.

### 2.6 Qdrant

Rust-based vector engine — performance and filtering focus.

| Strength | Limitation |
|----------|------------|
| Fast filtered search, good payloads | Operate cluster or pay cloud |
| REST/gRPC APIs | Smaller ecosystem than Pinecone |

**Best for:** Self-hosted performance-sensitive apps.

### 2.7 Milvus

Distributed vector database for very large collections.

| Strength | Limitation |
|----------|------------|
| Billions of vectors, sharding | Heavy infrastructure |
| GPU index options | Overkill for prototypes |

**Best for:** Large-scale production with dedicated platform team.

### 2.8 LanceDB

Embedded columnar store with vector search — serverless-friendly files.

**Best for:** Local/edge apps wanting file-based persistence without a server process.

---

## 3. Comparison Matrix

| System | Ops burden | Hybrid BM25 | SQL joins | Managed option | Learning curve |
|--------|------------|-------------|-----------|----------------|----------------|
| **Chroma** | Low | Manual/external | No | Limited | Low |
| **FAISS** | Low (lib) | Manual | No | No | Medium |
| **pgvector** | Medium (DBA) | Via Postgres FTS | **Yes** | RDS/cloud PG | Medium |
| **Pinecone** | Very low | Add-on / sparse | No | **Yes** | Low |
| **Weaviate** | Medium–high | **Built-in** | No | Yes | Medium |
| **Qdrant** | Medium | Payload + sparse vectors | No | Yes | Medium |
| **Milvus** | High | Varies | No | Zilliz Cloud | High |

Numbers in vendor benchmarks are **not** transferable — run notebook **09** on your corpus.

---

## 4. Decision Framework

Answer in order:

1. **Stage** — POC, MVP, or production at scale?
2. **Scale** — How many vectors? QPS? Growth in 12 months?
3. **Team** — Postgres skills? Dedicated platform engineers?
4. **Features** — Hybrid search? Multi-tenant filters? On-prem only?
5. **Budget** — Self-host cost vs managed per-million vectors

| Profile | Lean toward |
|---------|-------------|
| Learning / course / hackathon | **Chroma** |
| MVP on Postgres already | **pgvector** |
| No ops, need uptime SLA | **Pinecone** / managed |
| Error codes + SKUs matter | **Hybrid** (Weaviate, ES, pgvector FTS) |
| Research / custom index | **FAISS** |
| 10M+ vectors, self-hosted | **Qdrant** / **Milvus** |
| Air-gapped enterprise | Self-hosted **Qdrant** / **Milvus** / **Weaviate** |

**Default path for this curriculum:** Chroma → eval → pgvector or managed if scale/ops demand.

---

## 5. Demonstration: Weighted Decision Scorer

Simple rubric — adjust weights for your organization.

In [ ]:
REQUIREMENTS = {
    "low_ops": 9,           # 0-10 importance
    "hybrid_search": 6,
    "sql_joins": 8,
    "scale_millions": 4,
    "on_prem": 7,
}

OPTIONS = {
    "chroma": {"low_ops": 9, "hybrid_search": 3, "sql_joins": 1, "scale_millions": 4, "on_prem": 8},
    "pgvector": {"low_ops": 6, "hybrid_search": 6, "sql_joins": 10, "scale_millions": 6, "on_prem": 9},
    "pinecone": {"low_ops": 10, "hybrid_search": 5, "sql_joins": 1, "scale_millions": 9, "on_prem": 1},
    "qdrant": {"low_ops": 5, "hybrid_search": 7, "sql_joins": 1, "scale_millions": 8, "on_prem": 9},
}


def score_option(option_scores: dict[str, int], requirements: dict[str, int]) -> float:
    total_weight = sum(requirements.values())
    return sum(option_scores[k] * requirements[k] for k in requirements) / total_weight


ranked = sorted(OPTIONS.items(), key=lambda x: score_option(x[1], REQUIREMENTS), reverse=True)
print("Requirements:", REQUIREMENTS)
print("\nRanked options:")
for name, scores in ranked:
    print(f"  {name:10s}  weighted_score={score_option(scores, REQUIREMENTS):.2f}")

Change `REQUIREMENTS` to reflect your project — the winner shifts quickly when `sql_joins` or `on_prem` dominate.

---

## 6. Hybrid Search

### 6.1 Why Hybrid?

| Query type | Vector-only | Keyword (BM25) | Hybrid |
|------------|-------------|----------------|--------|
| "password reset flow" | Strong | Moderate | Strong |
| "ECONNREFUSED" | Weak | **Strong** | **Strong** |
| "SKU-XJ-9921" | Weak | **Strong** | **Strong** |
| Paraphrased policy question | **Strong** | Moderate | **Strong** |

**Dense** retrieval: embedding similarity. **Sparse** retrieval: BM25 / FTS on exact tokens.

### 6.2 Reciprocal Rank Fusion (RRF)

Merge two ranked lists without normalizing incompatible scores:

$$\text{RRF}(d) = \sum_{\text{lists } L} \frac{1}{k + \text{rank}_L(d)}$$

Common $k = 60$. Documents ranking well in **both** lists rise to the top.

In [ ]:
def reciprocal_rank_fusion(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    """rankings: list of doc-id lists, best-first per list."""
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


vector_ranking = ["doc_auth", "doc_migration", "doc_faq"]
keyword_ranking = ["doc_error_code", "doc_auth", "doc_migration"]

print("Vector ranking:", vector_ranking)
print("Keyword ranking:", keyword_ranking)
print("\nRRF merged:")
for doc_id, score in reciprocal_rank_fusion([vector_ranking, keyword_ranking]):
    print(f"  {score:.5f}  {doc_id}")

`doc_auth` appears in both lists and wins — typical hybrid behavior. Implement hybrid when pure vector Recall@k plateaus on keyword-heavy queries (notebook 09).

---

## 7. Cost and Operations Trade-offs

| Cost driver | Self-hosted | Managed |
|-------------|-------------|--------|
| Engineering time | Cluster care, upgrades | Lower |
| Infrastructure | VMs, K8s, storage | Per-vector / per-query pricing |
| Embedding API | Same either way | Same |
| Re-index jobs | Your schedulers | Your schedulers |

**Hidden costs:** observability, backups, disaster recovery, multi-region replication, support contracts.

At **small scale**, engineer time dominates — Chroma or pgvector wins. At **large scale**, managed pricing vs SRE headcount becomes the trade-off.

---

## 8. Migration Path: Chroma → Production Store

```
1. Export chunks + metadata from source of truth (not only Chroma)
2. Define VectorStore interface in app code
3. Implement Chroma adapter (today)
4. Run parallel index on pgvector / Qdrant / Pinecone
5. Compare Recall@k (notebook 09)
6. Feature-flag query traffic
7. Decommission old index
```

Never migrate by screen-scraping vectors only — always retain **canonical text** and **re-embed** if the target store uses a different metric or model.

---

## 9. Portability: VectorStore Interface

Abstract retrieval so RAG logic stays stable:

In [ ]:
@dataclass
class Hit:
    id: str
    text: str
    score: float
    metadata: dict[str, Any]


class VectorStore(Protocol):
    def upsert(self, ids: list[str], texts: list[str], embeddings: list[list[float]], metadatas: list[dict]) -> None: ...
    def search(self, query_embedding: list[float], k: int = 5, filters: dict | None = None) -> list[Hit]: ...
    def delete(self, ids: list[str]) -> None: ...


class ChromaVectorStore:
    """Thin adapter — swap implementation for pgvector/Qdrant later."""

    def __init__(self, collection):
        self._col = collection

    def upsert(self, ids, texts, embeddings, metadatas):
        self._col.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)

    def search(self, query_embedding, k=5, filters=None):
        kwargs = {"query_embeddings": [query_embedding], "n_results": k, "include": ["documents", "metadatas", "distances"]}
        if filters:
            kwargs["where"] = filters
        r = self._col.query(**kwargs)
        hits = []
        for i, doc_id in enumerate(r["ids"][0]):
            dist = r["distances"][0][i]
            hits.append(Hit(id=doc_id, text=r["documents"][0][i], score=-dist, metadata=r["metadatas"][0][i]))
        return hits

    def delete(self, ids):
        self._col.delete(ids=ids)


print("VectorStore protocol + ChromaVectorStore adapter defined.")

RAG services depend on `VectorStore`, not on Chroma imports directly — migration becomes an adapter swap.

---

## 10. Deployment Scenarios

| Scenario | Suggested stack |
|----------|-----------------|
| Solo dev RAG tutorial | Chroma persistent + OpenAI embed |
| Startup on Supabase/Neon | **pgvector** + existing Postgres |
| Enterprise on-prem | Qdrant / Milvus self-hosted |
| Variable traffic SaaS | Managed Pinecone/Qdrant Cloud |
| Heavy keyword + semantic | Weaviate or Elasticsearch hybrid |
| Batch analytics only | FAISS offline + parquet metadata |

---

## 11. When Not to Use a Vector DB

| Situation | Alternative |
|-----------|-------------|
| < 500 chunks, static | NumPy brute force in memory |
| Exact key lookup only | SQL/Redis |
| Graph relationships primary | Graph DB + optional vectors |
| Real-time OLTP on vectors | Reconsider architecture |

Vector DBs optimize **similarity search at scale** — not every feature needs one.

---

## 12. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Choose from benchmark leaderboard | Wrong for your domain | Eval notebook 09 |
| Milvus for 5k chunks | Ops overhead | Chroma/pgvector |
| Skip hybrid for SKU search | Miss exact matches | BM25 + RRF |
| Vendor lock without adapter | Painful migration | VectorStore interface |
| Ignore on-prem/compliance | Blocked deployment | Filter options early |
| Copy vectors only on migrate | Metric/model mismatch | Re-embed from text |

---

## 13. Summary

The vector database landscape spans **embedded** (Chroma, FAISS, LanceDB), **SQL** (pgvector), **servers** (Qdrant, Weaviate, Milvus), and **managed** clouds (Pinecone). Choose based on scale, ops, hybrid needs, and existing infrastructure — not hype.

Start with **Chroma** in this curriculum; use a **decision rubric** and **Recall@k** evals before migrating. **Hybrid search** (RRF) helps keyword-heavy queries. **Abstract VectorStore** in code to limit lock-in.

Next: **09. Evaluating Retrieval Quality** — measure before you migrate vendors.